# SABER Excytin **Insane Mode** Evaluation Analysis

## Purpose

**Self-contained** analysis notebook for the **Excytin** domain running in **insane mode** — where the 4 core security tables (`SecurityAlert`, `AlertEvidence`, `AlertInfo`, `SecurityIncident`) are **stripped** from all incident databases.

**What is insane mode?** It tests whether agents can still investigate security incidents when the primary alert/evidence tables are removed, forcing them to rely on secondary data sources (process logs, network connections, file events, etc.). This measures how dependent model performance is on the "easy path" of directly querying alert tables.

### Expected behavior

- **Scores should drop significantly** compared to normal mode, especially for tasks that directly ask about alerts, evidence, or incident details
- Models that relied heavily on SecurityAlert/AlertEvidence tables in normal mode will show the largest drops
- Tasks about file hashes, process execution, and network activity (available in remaining tables) may still be solvable

### Analyses included

| # | Analysis | Domain-Agnostic? |
|---|----------|------------------|
| 1 | Overall Reward Distribution | ✓ Generic |
| 2 | **Per-Incident Breakdown** | **Excytin-specific** |
| 3 | Cost Analysis & Pareto | ✓ Generic |
| 4 | Token Usage Breakdown | ✓ Generic |
| 5 | Cost Efficiency (Reward/$) | ✓ Generic |
| 6 | Sub-Task Decomposition | ✓ Generic |
| 7 | Checkpoint Distributions | ✓ Generic |
| 8 | **Submission vs. Checkpoint Gap Heatmap** | **Excytin-specific** |
| 9 | Agent Trajectory Analysis | ✓ Generic |
| 9a | Score vs. Tool-Call Limit | ✓ Generic |
| 9b | Effort Distribution | ✓ Generic |
| 10 | Tool Usage Analysis | ✓ Generic |
| 11 | Time Efficiency & Reward Per Tool Call | ✓ Generic |
| 12 | Score Outcome Segmentation by Effort | ✓ Generic |
| 13 | Cross-Model Difficulty Agreement | ✓ Generic |
| 14 | **SQL Query Analysis** | **Excytin-specific** |

### Sample models

| Model | API ID | Provider |
|-------|--------|----------|
| Claude Haiku 4.5 | `anthropic/claude-haiku-4-5` | Anthropic |
| Claude Sonnet 4.6 | `anthropic/claude-sonnet-4-6` | Anthropic |
| Claude Opus 4.6 | `anthropic/claude-opus-4-6` | Anthropic |
| Claude Strudel EAP | `anthropic/claude-strudel-eap` | Anthropic |
| GPT-5.4 | `openai/azure/gpt-5.4` | Azure OpenAI |
| GPT-5.4-mini | `openai/azure/gpt-5.4-mini-2026-03-17` | Azure OpenAI |
| Alpine Alpha | `openai/alpine-alpha` | OpenAI |
| Glacier Alpha | `openai/glacier-alpha` | OpenAI |

## Prerequisites & Running Insane Mode Evaluations

```bash
# Install
uv sync --all-extras
cp .env.template .env  # Configure API keys

# Quick insane mode test (1 sample)
uv run inspect eval domains/excytin --model anthropic/claude-haiku-4-5 \
  -T mode=insane -T dataset=latest_test_set --limit 1

# Full insane mode evaluation (599 samples)
uv run inspect eval domains/excytin --model anthropic/claude-sonnet-4-6 \
  -T mode=insane -T dataset=latest_test_set

# Batch run all models
bash latest_experiments/run_excytin_insane_batch.sh
```

### What `mode=insane` does

1. Runs the `GenerateInsaneSQL` setup hook which strips 4 tables from each incident's SQL file:
   - `SecurityAlert`, `AlertEvidence`, `AlertInfo`, `SecurityIncident`
2. Uses `all_incidents_insane.compose.yml` which mounts `*_insane.sql` files
3. All other tables (process events, network connections, file events, etc.) remain intact

### Viewing results

- **VS Code:** Install the **Inspect AI** extension → click any `.eval` file
- **Browser:** `uv run inspect view` → http://localhost:7575

<a id="configuration"></a>
## Configuration

Edit the cell below to add your own models. Insane mode eval files use the naming convention `{model}_latest_test_set_insane.eval`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Excytin domain (INSANE MODE)
# ══════════════════════════════════════════════════════════════════════════════
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from inspect_ai.log import read_eval_log

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from saber_analysis.data_loader import load_eval_logs, load_baseline_logs, ensure_eval_files, load_trajectory_data
from saber_analysis.cost import extract_cost_rows
from saber_analysis.plots import setup_plotting, classify_score_type

setup_plotting()

DOMAIN_NAME = 'Excytin (Insane Mode)'
DOMAIN_SLUG = 'excytin'
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
LOG_DIR = str(REPO_ROOT / 'eval_samples' / DOMAIN_SLUG)
ARTIFACTS_DIR = NOTEBOOK_DIR / 'artifacts' / 'excytin_insane'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

EVAL_LOGS = {
    'Claude Haiku 4.5':    'haiku_4.5_latest_test_set_insane.eval',
    'Claude Sonnet 4.6':   'sonnet_4.6_latest_test_set_insane.eval',
    'Claude Opus 4.6':     'opus_4.6_latest_test_set_insane.eval',
    'Claude Strudel EAP':  'strudel_eap_latest_test_set_insane.eval',
    'GPT-5.4':             'gpt_5.4_latest_test_set_insane.eval',
    'GPT-5.4-mini':        'gpt_5.4_mini_latest_test_set_insane.eval',
    'Alpine Alpha':        'alpine_alpha_latest_test_set_insane.eval',
    'Glacier Alpha':       'glacier_alpha_latest_test_set_insane.eval',
}

NO_THINKING_LOGS = {
    'Claude Sonnet 4.6':   'sonnet_4.6_latest_test_set_insane_no_thinking.eval',
    'Claude Opus 4.6':     'opus_4.6_latest_test_set_insane_no_thinking.eval',
    'Claude Strudel EAP':  'strudel_eap_latest_test_set_insane_no_thinking.eval',
    'GPT-5.4':             'gpt_5.4_latest_test_set_insane_no_reasoning.eval',
    'GPT-5.4-mini':        'gpt_5.4_mini_latest_test_set_insane_no_reasoning.eval',
    'Alpine Alpha':        'alpine_alpha_latest_test_set_insane_no_reasoning.eval',
    'Glacier Alpha':       'glacier_alpha_latest_test_set_insane_no_reasoning.eval',
}

COLORS = {
    'Claude Haiku 4.5':    '#17BECF',  # cyan
    'Claude Sonnet 4.6':   '#F39C12',  # orange
    'Claude Opus 4.6':     '#E74C3C',  # red
    'Claude Strudel EAP':  '#7B68EE',  # medium slate blue
    'GPT-5.4':             '#2CA02C',  # green
    'GPT-5.4-mini':        '#7B4FBF',  # purple
    'Alpine Alpha':        '#FF6B6B',  # coral
    'Glacier Alpha':       '#1ABC9C',  # teal
}

N_SAMPLES = 599
MODEL_ORDER = list(EVAL_LOGS.keys())

# Excytin groups by security incident
GROUP_FN = lambda sid: '_'.join(sid.split('_')[:2])
GROUP_LABEL = 'Incident'

# NOTE: Strudel EAP and Alpine Alpha do not have published pricing.
# Strudel EAP uses Opus-tier pricing as a proxy; Alpine Alpha uses GPT-5.4-tier pricing.
PRICING = {
    'anthropic/claude-haiku-4-5':            {'input': 0.80,  'output': 4.00,  'cache_read': 0.08,   'cache_write': 1.00},
    'anthropic/claude-sonnet-4-6':           {'input': 3.00,  'output': 15.00, 'cache_read': 0.375,  'cache_write': 3.75},
    'anthropic/claude-opus-4-6':             {'input': 15.00, 'output': 75.00, 'cache_read': 1.875,  'cache_write': 18.75},
    'anthropic/claude-strudel-eap':          {'input': 15.00, 'output': 75.00, 'cache_read': 1.875,  'cache_write': 18.75},
    'openai/azure/gpt-5.4':                  {'input': 10.00, 'output': 30.00, 'cache_read': 2.50,   'cache_write': 0.0},
    'openai/azure/gpt-5.4-mini-2026-03-17':  {'input': 1.10,  'output': 4.40,  'cache_read': 0.275,  'cache_write': 0.0},
    'openai/alpine-alpha':                   {'input': 10.00, 'output': 30.00, 'cache_read': 2.50,   'cache_write': 0.0},
    'openai/glacier-alpha':                   {'input': 10.00, 'output': 30.00, 'cache_read': 2.50,   'cache_write': 0.0},
}

ensure_eval_files(
    EVAL_LOGS, LOG_DIR, DOMAIN_SLUG,
    baseline_logs=NO_THINKING_LOGS,
    fallback_dirs=[str(REPO_ROOT / 'latest_experiments' / DOMAIN_SLUG)],
)

# Auto-detect tool-call limit from eval log headers
_first_log_path = os.path.join(LOG_DIR, next(iter(EVAL_LOGS.values())))
_log_header = read_eval_log(_first_log_path, header_only=True)
TOOL_CALL_LIMIT = _log_header.eval.config.tool_call_limit or 25
TIME_LIMIT = _log_header.eval.config.time_limit

print(f'✓ Domain: {DOMAIN_NAME}')
print(f'  Mode: INSANE (SecurityAlert, AlertEvidence, AlertInfo, SecurityIncident stripped)')
print(f'  Models: {len(EVAL_LOGS)} ({", ".join(MODEL_ORDER)})')
print(f'  Samples: {N_SAMPLES}')
print(f'  Tool-call limit: {TOOL_CALL_LIMIT} (from eval header)')
print(f'  Time limit: {TIME_LIMIT}s' if TIME_LIMIT else '  Time limit: none')
print(f'  Log dir: {LOG_DIR}')

## Data Loading

Parses all `.eval` ZIP archives and extracts per-sample scores, sub-task scores, group labels, and baseline scores. These DataFrames are the foundation for all 14 experiments.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════
samples_df, subtasks_df, overall_df, model_overall = load_eval_logs(
    EVAL_LOGS, LOG_DIR, MODEL_ORDER, group_fn=GROUP_FN
)
baseline_samples_df, baseline_subtasks_df, no_thinking_overall = load_baseline_logs(
    NO_THINKING_LOGS, LOG_DIR, group_fn=GROUP_FN
)

print(f'\n✓ Loaded {len(samples_df)} sample scores across {samples_df["model"].nunique()} models')
print(f'✓ Loaded {len(subtasks_df)} sub-task scores')
print(f'✓ Incidents: {sorted(samples_df["group"].unique())}')
print()
print('═══ Overall SABER Scores (insane mode, with extended reasoning) ═══')
for m in MODEL_ORDER:
    if m in model_overall:
        r = overall_df.loc[m]
        print(f'  {m:25s}  mean={r["mean"]:.4f}  stderr={r["stderr"]:.4f}')
print()
print('═══ Baselines (insane mode, without extended reasoning) ═══')
for m, r in no_thinking_overall.items():
    if m in model_overall:
        delta = model_overall[m]['mean'] - r['mean']
        print(f'  {m:25s}  baseline={r["mean"]:.4f}  reasoning={model_overall[m]["mean"]:.4f}  Δ={delta:+.4f}')

---

## 1. Overall Reward Distribution per Model (Insane Mode)

### What this measures

The **mean `saber_overall` score** across all 599 test samples for each model, with security alert tables stripped.

### Key questions for insane mode

- How much does each model's score **drop** compared to normal mode?
- Which models are most resilient to missing alert data?
- Do models that scored highest in normal mode also lead in insane mode?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 1: Overall mean scores (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(13, 6.5))
means = [overall_df.loc[m, 'mean'] for m in MODEL_ORDER]
stderrs = [overall_df.loc[m, 'stderr'] for m in MODEL_ORDER]
colors = [COLORS[m] for m in MODEL_ORDER]

base_heights, delta_heights = [], []
for m in MODEL_ORDER:
    if m in no_thinking_overall:
        base_heights.append(no_thinking_overall[m]['mean'])
        delta_heights.append(overall_df.loc[m, 'mean'] - no_thinking_overall[m]['mean'])
    else:
        base_heights.append(overall_df.loc[m, 'mean'])
        delta_heights.append(0.0)

x = np.arange(len(MODEL_ORDER))
ax.bar(x, base_heights, color=colors, edgecolor='black', linewidth=0.5, alpha=0.85)
for i, m in enumerate(MODEL_ORDER):
    if delta_heights[i] != 0:
        ax.bar(i, abs(delta_heights[i]),
               bottom=base_heights[i] if delta_heights[i] > 0 else base_heights[i] + delta_heights[i],
               color=colors[i], edgecolor='black', linewidth=0.5, alpha=0.45, hatch='///')
ax.errorbar(x, means, yerr=stderrs, fmt='none', ecolor='black', capsize=5, linewidth=1.2)
for i, (mean, se) in enumerate(zip(means, stderrs)):
    ax.text(i, mean + se + 0.012, f'{mean:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
for i, m in enumerate(MODEL_ORDER):
    if m in no_thinking_overall:
        d = delta_heights[i]
        ax.text(i, base_heights[i] + d/2, f'{d:+.4f}', ha='center', va='center', fontsize=9, fontweight='bold',
                color='#006400' if d > 0 else '#8B0000',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.8, edgecolor='none'))

ax.set_xticks(x); ax.set_xticklabels(MODEL_ORDER, fontsize=10, rotation=15, ha='right')
ax.set_ylabel('Mean SABER Score (saber_overall)')
ax.set_title(f'Overall Reward per Model — Excytin INSANE MODE ({N_SAMPLES} samples)', fontweight='bold')
ax.set_ylim(0, 1.10); ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.legend(handles=[mpatches.Patch(facecolor='gray', edgecolor='black', alpha=0.85, label='Default'),
                   mpatches.Patch(facecolor='gray', edgecolor='black', alpha=0.45, hatch='///', label='Δ extended reasoning')],
          loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'overall_reward_insane.png', bbox_inches='tight')
plt.show()

---

## 2. Per-Incident Reward Breakdown (Insane Mode)

### Key questions

- Which incidents are most affected by stripping alert tables?
- Are some incidents still solvable without SecurityAlert/AlertEvidence data?
- Do incident difficulty rankings change compared to normal mode?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 2: Per-incident grouped bar chart (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
group_scores = samples_df.groupby(['group', 'model'])['score'].mean().unstack('model').reindex(columns=MODEL_ORDER).sort_index()
group_counts = samples_df[samples_df['model'] == MODEL_ORDER[0]].groupby('group').size()

baseline_group = pd.DataFrame()
if len(baseline_samples_df) > 0:
    baseline_group = baseline_samples_df.groupby(['group', 'model'])['score'].mean().unstack('model')
    baseline_group = baseline_group.reindex(index=group_scores.index, columns=MODEL_ORDER)

print('═══ Per-Incident Mean Scores (insane mode) ═══')
print(group_scores.round(4).to_string())

fig, ax = plt.subplots(figsize=(18, 7))
n_groups, n_models = len(group_scores), len(MODEL_ORDER)
bar_width = 0.15
xg = np.arange(n_groups)

for i, model in enumerate(MODEL_ORDER):
    offset = (i - n_models/2 + 0.5) * bar_width
    full_vals = group_scores[model].values
    has_base = model in no_thinking_overall and len(baseline_group) > 0 and model in baseline_group.columns

    if has_base:
        base_vals = baseline_group[model].values
        ax.bar(xg + offset, base_vals, bar_width, color=COLORS[model], edgecolor='black', linewidth=0.3, alpha=0.85)
        for xi in range(len(xg)):
            d = full_vals[xi] - base_vals[xi]
            if abs(d) > 0.001:
                ax.bar(xg[xi] + offset, abs(d), bottom=base_vals[xi] if d > 0 else full_vals[xi],
                       width=bar_width, color=COLORS[model], edgecolor='black', linewidth=0.3, alpha=0.45, hatch='///')
    else:
        ax.bar(xg + offset, full_vals, bar_width, color=COLORS[model], edgecolor='black', linewidth=0.3, alpha=0.85)

    for xi, v in zip(xg, full_vals):
        ax.text(xi + offset, v + 0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=6, fontweight='bold')

handles = [mpatches.Patch(facecolor=COLORS[m], edgecolor='black', alpha=0.85, label=m) for m in MODEL_ORDER]
handles.append(mpatches.Patch(facecolor='gray', edgecolor='black', alpha=0.45, hatch='///', label='Δ extended reasoning'))
ax.legend(handles=handles, loc='upper right', fontsize=8, framealpha=0.9, ncol=2)
ax.set_xticks(xg)
ax.set_xticklabels([f'{g}\n({group_counts.get(g, "?")} tasks)' for g in group_scores.index], fontsize=9)
ax.set_ylabel('Mean SABER Score')
ax.set_title(f'Per-Incident SABER Scores — Excytin INSANE MODE ({N_SAMPLES} samples)', fontweight='bold')
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'per_incident_scores_insane.png', bbox_inches='tight')
plt.show()

---

## 3. Cost Analysis (Insane Mode)

How much does insane mode cost compared to normal mode? Models may use more tokens searching for missing data.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 3a: Cost extraction (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
cost_df = pd.DataFrame(extract_cost_rows(EVAL_LOGS, LOG_DIR, PRICING, N_SAMPLES)).set_index('model').reindex(MODEL_ORDER)
_baseline_rows = extract_cost_rows(NO_THINKING_LOGS, LOG_DIR, PRICING, N_SAMPLES)
baseline_cost_df = pd.DataFrame(_baseline_rows).set_index('model') if _baseline_rows else pd.DataFrame()

print('═══ Cost (insane mode, with extended reasoning) ═══')
print(cost_df[['score', 'total_cost', 'cost_per_sample']].round(2).to_string())
print()
print('═══ Cost (insane mode, default / no reasoning) ═══')
print(baseline_cost_df[['score', 'total_cost', 'cost_per_sample']].round(2).to_string() if len(baseline_cost_df) > 0 else '(no baseline logs configured)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 3b: Cost visualization (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
x = np.arange(len(MODEL_ORDER))

for panel, (col, ylabel, title, fmt) in enumerate([
    ('total_cost', 'Estimated Cost ($)', f'Total Eval Cost ({N_SAMPLES} samples)', '${:.0f}'),
    ('cost_per_sample', 'Cost per Sample ($)', 'Cost per Sample', '${:.3f}'),
]):
    ax = axes[panel]
    for i, m in enumerate(MODEL_ORDER):
        val = cost_df.loc[m, col]
        if m in baseline_cost_df.index:
            base = baseline_cost_df.loc[m, col]
            ax.bar(i, base, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.85)
            d = val - base
            ax.bar(i, abs(d), bottom=base if d > 0 else val, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.45, hatch='///')
        else:
            ax.bar(i, val, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.85)
        ax.text(i, val + cost_df[col].max()*0.02, fmt.format(val), ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_xticks(x); ax.set_xticklabels(MODEL_ORDER, fontsize=8, rotation=20, ha='right')
    ax.set_ylabel(ylabel); ax.set_title(title, fontweight='bold')

ax = axes[2]
for m in MODEL_ORDER:
    row = cost_df.loc[m]
    ax.scatter(row['total_cost'], row['score'], s=200, color=COLORS[m], edgecolor='black', linewidth=1, zorder=5)
    ax.annotate(m, (row['total_cost'], row['score']), textcoords='offset points', xytext=(8, 8), fontsize=9, fontweight='bold')
for m in baseline_cost_df.index:
    if m not in COLORS: continue
    brow = baseline_cost_df.loc[m]
    ax.scatter(brow['total_cost'], brow['score'], s=200, facecolors='none', edgecolors=COLORS[m], linewidth=2.5, zorder=4)
    ax.annotate('', xy=(cost_df.loc[m, 'total_cost'], cost_df.loc[m, 'score']), xytext=(brow['total_cost'], brow['score']),
                arrowprops=dict(arrowstyle='->', color=COLORS[m], lw=1.5, alpha=0.5))
ax.set_xlabel('Total Cost ($)'); ax.set_ylabel('Mean SABER Score'); ax.set_title('Score vs. Cost (Pareto)', fontweight='bold')

fig.suptitle(f'Cost Analysis — Excytin INSANE MODE ({N_SAMPLES} samples)', fontweight='bold', fontsize=14, y=1.02)
fig.text(0.5, -0.04,
    'Hatched bars (///) show the additional cost from enabling reasoning.\n'
    'Pricing note: Strudel EAP uses Opus-tier pricing as a proxy; '
    'Alpine Alpha and Glacier Alpha use GPT-5.4-tier pricing.',
    ha='center', va='top', fontsize=9, fontstyle='italic', color='#555555', wrap=True)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'cost_analysis_insane.png', bbox_inches='tight')
plt.show()

---

## 4. Token Usage Breakdown (Insane Mode)

Do models use more tokens when alert tables are missing — searching fruitlessly for data that doesn't exist?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 4: Token usage stacked horizontal bars (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
categories = ['Input', 'Output', 'Cache Read', 'Cache Write', 'Reasoning']
cat_colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']
token_keys = ['input_tokens', 'output_tokens', 'cache_read', 'cache_write', 'reasoning_tokens']

fig, ax = plt.subplots(figsize=(18, 7.5))
bar_height = 0.55
all_totals = [sum(cost_df.loc[m][k]/1e6 for k in token_keys) for m in MODEL_ORDER]
global_max = max(all_totals)
MIN_LABEL_WIDTH = global_max * 0.05

for i, m in enumerate(MODEL_ORDER):
    row = cost_df.loc[m]
    vals = [row[k]/1e6 for k in token_keys]
    has_baseline = m in baseline_cost_df.index
    bvals = [baseline_cost_df.loc[m][k]/1e6 for k in token_keys] if has_baseline else vals

    left = 0
    for j, (rv, bv, c) in enumerate(zip(vals, bvals, cat_colors)):
        if has_baseline and rv > 0:
            solid_w = min(rv, bv)
            ax.barh(i, solid_w, left=left, height=bar_height, color=c, edgecolor='white',
                    linewidth=0.5, alpha=0.85, label=categories[j] if i == 0 else '')
            if rv - bv > 0.01:
                ax.barh(i, rv - bv, left=left + solid_w, height=bar_height, color=c,
                        edgecolor='white', linewidth=0.5, alpha=0.45, hatch='///')
        else:
            ax.barh(i, rv, left=left, height=bar_height, color=c, edgecolor='white',
                    linewidth=0.5, alpha=0.85, label=categories[j] if i == 0 else '')
        if rv >= MIN_LABEL_WIDTH:
            ax.text(left + rv/2, i, f'{rv:.1f}M', ha='center', va='center', fontsize=7, fontweight='bold')
        left += rv

    ax.text(left + 0.5, i, f'{sum(vals):.1f}M', va='center', ha='left', fontsize=9, fontweight='bold')
    if has_baseline:
        bt = sum(bvals); td = sum(vals) - bt
        ax.text(left + 0.5, i - 0.22, f'default: {bt:.1f}M  (Δ {td:+.1f}M / {td/bt*100:+.0f}%)',
                va='center', ha='left', fontsize=7.5, fontstyle='italic', color='#555')

ax.set_yticks(range(len(MODEL_ORDER))); ax.set_yticklabels(MODEL_ORDER, fontsize=10)
ax.set_xlabel('Tokens (millions)'); ax.set_title('Token Usage Breakdown — Excytin INSANE MODE', fontweight='bold')
ax.set_xlim(right=global_max * 1.35); ax.set_ylim(-0.6, len(MODEL_ORDER) - 0.3)
legend_handles = [mpatches.Patch(facecolor=c, edgecolor='white', alpha=0.85, label=cat) for cat, c in zip(categories, cat_colors)]
legend_handles += [mpatches.Patch(facecolor='gray', edgecolor='black', alpha=0.85, label='Default'),
                   mpatches.Patch(facecolor='gray', edgecolor='black', alpha=0.45, hatch='///', label='Δ extended reasoning')]
ax.legend(handles=legend_handles, loc='lower right', fontsize=8.5)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'token_usage_insane.png', bbox_inches='tight')
plt.show()

---

## 5. Cost Efficiency: Reward per Dollar (Insane Mode)

$$\text{Cost Efficiency} = \frac{\text{Mean SABER Score}}{\text{Total Cost (\$)}}$$

With lower scores in insane mode, cost efficiency may shift — models that maintain scores while not burning extra tokens searching will be most efficient.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 5: Cost efficiency (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 6))
rpd = cost_df['score'] / cost_df['total_cost']
base_rpd = baseline_cost_df['score'] / baseline_cost_df['total_cost'] if len(baseline_cost_df) > 0 else pd.Series(dtype=float)

for i, m in enumerate(MODEL_ORDER):
    val = rpd[m]
    if m in base_rpd.index:
        bv = base_rpd[m]
        ax.bar(i, bv, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.85)
        d = val - bv
        ax.bar(i, abs(d), bottom=bv if d > 0 else val, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.45, hatch='///')
    else:
        ax.bar(i, val, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.text(i, val + rpd.max()*0.02, f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_xticks(range(len(MODEL_ORDER))); ax.set_xticklabels(MODEL_ORDER, fontsize=10, rotation=15, ha='right')
ax.set_ylabel('Score / Cost ($)'); ax.set_title('Cost Efficiency: Reward per Dollar — Excytin INSANE MODE', fontweight='bold', pad=15)
ax.set_ylim(0, rpd.max() * 1.25)
ax.legend(handles=[mpatches.Patch(facecolor='gray', edgecolor='black', alpha=0.85, label='Default'),
                   mpatches.Patch(facecolor='gray', edgecolor='black', alpha=0.45, hatch='///', label='Δ extended reasoning')],
          loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'cost_efficiency_insane.png', bbox_inches='tight')
plt.show()

print('═══ Cost Efficiency Summary (insane mode) ═══')
print(f'{"Model":25s} {"Score":>6s} {"Cost":>8s} {"$/sample":>10s} {"Score/$":>10s}')
print('─' * 60)
for m in MODEL_ORDER:
    row = cost_df.loc[m]
    print(f'{m:25s} {row["score"]:6.3f} ${row["total_cost"]:7.2f} ${row["cost_per_sample"]:9.4f} {rpd[m]:10.5f}')

---

## 6. Sub-Task Score Breakdown (Insane Mode)

Submission (final answer) + checkpoint (intermediate investigation steps) + aggregate scores. In insane mode, checkpoints requiring alert table queries should drop to near zero.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 6: Sub-task breakdown (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
subtasks_df['category'] = subtasks_df['score_type'].apply(classify_score_type)
cat_means = subtasks_df.groupby(['model', 'category'])['score'].mean().unstack('category')
available_cats = [c for c in ['Submission', 'Checkpoints', 'Aggregate'] if c in cat_means.columns]
cat_means = cat_means.reindex(MODEL_ORDER)[available_cats]

baseline_cat_means = pd.DataFrame()
if len(baseline_subtasks_df) > 0:
    baseline_subtasks_df['category'] = baseline_subtasks_df['score_type'].apply(classify_score_type)
    baseline_cat_means = baseline_subtasks_df.groupby(['model', 'category'])['score'].mean().unstack('category')
    baseline_cat_means = baseline_cat_means.reindex(columns=available_cats)

print('═══ Sub-Task Mean Scores (insane mode) ═══')
print(cat_means.round(4).to_string())

fig, ax = plt.subplots(figsize=(14, 6))
cat_plot_colors = ['#3498DB', '#E67E22', '#2ECC71'][:len(available_cats)]
x = np.arange(len(MODEL_ORDER))
bar_width = 0.22
for j, (cat, color) in enumerate(zip(available_cats, cat_plot_colors)):
    offset = (j - len(available_cats)/2 + 0.5) * bar_width
    for i, m in enumerate(MODEL_ORDER):
        fv = cat_means.loc[m, cat]
        has_base = (len(baseline_cat_means) > 0 and m in baseline_cat_means.index
                    and cat in baseline_cat_means.columns and not pd.isna(baseline_cat_means.loc[m, cat]))
        if has_base:
            bv = baseline_cat_means.loc[m, cat]
            ax.bar(x[i]+offset, bv, bar_width, color=color, edgecolor='black', linewidth=0.3, alpha=0.85)
            d = fv - bv
            if abs(d) > 0.001:
                ax.bar(x[i]+offset, abs(d), bottom=bv if d > 0 else fv, width=bar_width, color=color,
                       edgecolor='black', linewidth=0.3, alpha=0.45, hatch='///')
        else:
            ax.bar(x[i]+offset, fv, bar_width, color=color, edgecolor='black', linewidth=0.3, alpha=0.85)
        ax.text(x[i]+offset, fv+0.01, f'{fv:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

handles = [mpatches.Patch(facecolor=c, edgecolor='black', alpha=0.85, label=cat) for cat, c in zip(available_cats, cat_plot_colors)]
handles.append(mpatches.Patch(facecolor='gray', edgecolor='black', alpha=0.45, hatch='///', label='Δ extended reasoning'))
ax.legend(handles=handles, fontsize=9, loc='lower right')
ax.set_xticks(x); ax.set_xticklabels(MODEL_ORDER, fontsize=10, rotation=15, ha='right')
ax.set_ylabel('Mean Score'); ax.set_title('Sub-Task Breakdown — Excytin INSANE MODE', fontweight='bold'); ax.set_ylim(0, 1.12)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'subtask_breakdown_insane.png', bbox_inches='tight')
plt.show()

---

## 7. Checkpoint Score Distributions (Insane Mode)

With alert tables stripped, checkpoint distributions should shift heavily toward 0.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 7: Checkpoint distributions (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
checkpoint_df = subtasks_df[subtasks_df['score_type'].str.startswith('checkpoint')].copy()
fig, ax = plt.subplots(figsize=(14, 6))
positions, tick_labels = [], []
for i, model in enumerate(MODEL_ORDER):
    data = checkpoint_df[checkpoint_df['model'] == model]['score'].values
    if len(data) == 0: continue
    vp = ax.violinplot([data], positions=[i], showmeans=True, showmedians=True, widths=0.7)
    for pc in vp['bodies']: pc.set_facecolor(COLORS[model]); pc.set_alpha(0.7)
    for part in ('cmaxes','cmins','cbars','cmeans','cmedians'):
        if part in vp: vp[part].set_edgecolor('black'); vp[part].set_linewidth(0.8)
    ax.text(i, 1.05, f'μ={np.mean(data):.3f}\nn={len(data)}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    positions.append(i); tick_labels.append(model)

ax.set_xticks(positions); ax.set_xticklabels(tick_labels, fontsize=10, rotation=15, ha='right')
ax.set_ylabel('Checkpoint Score'); ax.set_title('Checkpoint Distributions — Excytin INSANE MODE', fontweight='bold')
ax.set_ylim(-0.05, 1.2)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'checkpoint_distributions_insane.png', bbox_inches='tight')
plt.show()

---

## 8. Submission vs. Checkpoint Gap Analysis (Insane Mode)

In insane mode, the gap should widen further — submissions may still partially succeed through alternative data paths, but checkpoints that specifically require alert data will fail.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 8 (EXCYTIN-SPECIFIC): Submission vs checkpoint gap heatmap (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
sub_means = subtasks_df[subtasks_df['category'] == 'Submission'].groupby(['group', 'model'])['score'].mean().unstack('model').reindex(columns=MODEL_ORDER)
cp_means = subtasks_df[subtasks_df['category'] == 'Checkpoints'].groupby(['group', 'model'])['score'].mean().unstack('model').reindex(columns=MODEL_ORDER)
gap = sub_means - cp_means

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(gap.T.values, cmap='RdYlGn_r', aspect='auto', vmin=-0.3, vmax=0.3)
ax.set_xticks(range(len(gap.index)))
ax.set_xticklabels(gap.index, fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(len(MODEL_ORDER)))
ax.set_yticklabels(MODEL_ORDER, fontsize=10)

for i, model in enumerate(MODEL_ORDER):
    for j, inc in enumerate(gap.index):
        val = gap.loc[inc, model]
        if not np.isnan(val):
            ax.text(j, i, f'{val:+.2f}', ha='center', va='center', fontsize=7,
                    fontweight='bold', color='white' if abs(val) > 0.15 else 'black')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Submission − Checkpoint Gap', fontsize=10)
ax.set_title('Submission vs Checkpoint Gap per Incident — INSANE MODE\n(positive = submission easier than checkpoints)', fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'submission_checkpoint_gap_insane.png', bbox_inches='tight')
plt.show()

print('═══ Average Submission–Checkpoint Gap per Model (insane mode) ═══')
for m in MODEL_ORDER:
    print(f'  {m:25s}  avg gap = {gap[m].mean():+.4f}')

---

## 9. Agent Trajectory Analysis (Insane Mode)

Do agents use more tool calls in insane mode, searching for the missing data? Or do they give up earlier?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAJECTORY DATA LOADING (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
from collections import Counter

traj_df = load_trajectory_data(EVAL_LOGS, LOG_DIR, MODEL_ORDER, group_fn=GROUP_FN)

# Normalize tool names to lowercase
traj_df['tool_counts'] = traj_df['tool_counts'].apply(
    lambda d: dict(Counter({k.lower(): v for k, v in d.items()}))
)

# Flatten tool_counts dicts into separate columns per tool
all_tools = sorted({t for tc in traj_df['tool_counts'] for t in tc})
for tool in all_tools:
    traj_df[f'tool_{tool}'] = traj_df['tool_counts'].apply(lambda d, t=tool: d.get(t, 0))

print(f'✓ Loaded trajectory data: {len(traj_df)} samples × {traj_df["model"].nunique()} models')
print(f'  Tools observed: {all_tools}')
print()
for m in MODEL_ORDER:
    mdf = traj_df[traj_df['model'] == m]
    print(f'  {m:25s}  steps={mdf.n_steps.mean():.1f}±{mdf.n_steps.std():.1f}  '
          f'tool_calls={mdf.n_tool_calls.mean():.1f}±{mdf.n_tool_calls.std():.1f}  '
          f'time={mdf.total_time.mean():.0f}s±{mdf.total_time.std():.0f}s')

### 9a. Mean Score vs. Tool-Call Limit (Insane Mode)

In insane mode the score ceiling is lower — the curve should plateau at a lower value and potentially rise more slowly as agents spend more effort searching.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 9a: Mean Score vs. Max Tool-Call Limit (insane mode)
# ══════════════════════════════════════════════════════════════════════════════

step_lookup = traj_df[['model', 'sample_id', 'n_tool_calls']].copy()
merged = subtasks_df.merge(step_lookup, on=['model', 'sample_id'], how='inner')
if 'category' not in merged.columns:
    merged['category'] = merged['score_type'].apply(classify_score_type)

call_limits = list(range(1, TOOL_CALL_LIMIT + 1))

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=False)

for ax, (score_label, score_src) in zip(axes, [
    ('Overall Score (saber_overall)', 'overall'),
    ('Checkpoint Score', 'checkpoint'),
]):
    for m in MODEL_ORDER:
        if score_src == 'overall':
            mdf = traj_df[traj_df['model'] == m].copy()
            total_samples = len(mdf)
        else:
            mdf = merged[(merged['model'] == m) & (merged['category'] == 'Checkpoints')].copy()
            if mdf.empty:
                continue
            total_samples = mdf['sample_id'].nunique()

        means = []
        for limit in call_limits:
            completed = mdf[mdf['n_tool_calls'] <= limit]
            avg = completed['score'].sum() / total_samples if total_samples > 0 else 0
            means.append(avg)

        ax.plot(call_limits, means, color=COLORS[m],
                linewidth=2.5, marker='o', markersize=3, label=m, zorder=5)

    ax.set_xlabel('Max Tool-Call Limit')
    ax.set_ylabel('Mean Score (all samples; unfinished = 0)')
    ax.set_title(score_label, fontweight='bold')
    ax.legend(fontsize=9, loc='best', framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(f'Mean Score vs. Tool-Call Limit — {DOMAIN_NAME} ({N_SAMPLES} samples per model)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'score_vs_tool_call_limit_insane.png', bbox_inches='tight')
plt.show()

# Summary: at what tool-call limit does each model reach 95% of its final score?
print('═══ Tool-Call Limit Analysis (insane mode, overall score) ═══')
print('Tool-call limit at which model reaches 95% of its full score:\n')
for m in MODEL_ORDER:
    mdf = traj_df[traj_df['model'] == m]
    total = len(mdf)
    full_score = mdf['score'].mean()
    target = full_score * 0.95
    limit_95 = TOOL_CALL_LIMIT
    for limit in call_limits:
        avg = mdf[mdf['n_tool_calls'] <= limit]['score'].sum() / total
        if avg >= target:
            limit_95 = limit
            break
    pct_done = len(mdf[mdf['n_tool_calls'] <= limit_95]) / total
    print(f'  {m:25s}  95% at {limit_95:2d} calls  '
          f'({pct_done:.0%} of samples done, full={full_score:.4f})')

### 9b. Effort Distribution per Model (Insane Mode)

Do models hit the tool-call limit more often in insane mode (searching for missing data)?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 9b: Tool call & step count distribution (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
if N_SAMPLES < 3:
    print(f'⚠ Only {N_SAMPLES} sample(s) per model — skipping violin plots (need ≥3 for meaningful distributions)')
    print()
    for m in MODEL_ORDER:
        mdf = traj_df[traj_df['model'] == m]
        print(f'  {m:25s}  tool_calls={mdf["n_tool_calls"].values}  steps={mdf["n_steps"].values}')
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # Panel 1: Tool call distribution
    ax = axes[0]
    for i, m in enumerate(MODEL_ORDER):
        data = traj_df[traj_df['model'] == m]['n_tool_calls'].values
        vp = ax.violinplot([data], positions=[i], showmeans=True, showmedians=True, widths=0.7)
        for pc in vp['bodies']:
            pc.set_facecolor(COLORS[m])
            pc.set_alpha(0.7)
        for part in ('cmaxes', 'cmins', 'cbars', 'cmeans', 'cmedians'):
            if part in vp:
                vp[part].set_edgecolor('black')
                vp[part].set_linewidth(0.8)
        ax.text(i, data.max() + 0.5, f'μ={np.mean(data):.1f}\nmed={np.median(data):.0f}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels(MODEL_ORDER, fontsize=9, rotation=15, ha='right')
    ax.set_ylabel('Tool Calls')
    ax.set_title('Tool Call Distribution per Model', fontweight='bold')

    # Panel 2: Step count distribution
    ax = axes[1]
    for i, m in enumerate(MODEL_ORDER):
        data = traj_df[traj_df['model'] == m]['n_steps'].values
        vp = ax.violinplot([data], positions=[i], showmeans=True, showmedians=True, widths=0.7)
        for pc in vp['bodies']:
            pc.set_facecolor(COLORS[m])
            pc.set_alpha(0.7)
        for part in ('cmaxes', 'cmins', 'cbars', 'cmeans', 'cmedians'):
            if part in vp:
                vp[part].set_edgecolor('black')
                vp[part].set_linewidth(0.8)
        ax.text(i, data.max() + 0.5, f'μ={np.mean(data):.1f}\nmed={np.median(data):.0f}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels(MODEL_ORDER, fontsize=9, rotation=15, ha='right')
    ax.set_ylabel('Steps (assistant turns)')
    ax.set_title('Step Count Distribution per Model', fontweight='bold')

    fig.suptitle(f'Agent Effort Distribution — {DOMAIN_NAME}', fontweight='bold', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / 'effort_distributions_insane.png', bbox_inches='tight')
    plt.show()

---

## 10. Tool Usage Analysis (Insane Mode)

Tool usage patterns — do models shift tool usage when alert tables are missing?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 10: Tool usage breakdown per model (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
tool_expanded = traj_df['tool_counts'].apply(pd.Series).fillna(0)
for col in tool_expanded.columns:
    traj_df[f'tool_{col}'] = tool_expanded[col]

tool_cols = [c for c in traj_df.columns if c.startswith('tool_') and c != 'tool_counts']
tool_names = [c.replace('tool_', '') for c in tool_cols]

# Aggregate minor tools (<1% of calls) into "Other"
total_calls_all = traj_df['n_tool_calls'].sum()
tool_totals = {tname: traj_df[col].sum() for col, tname in zip(tool_cols, tool_names)}
major_tools = [t for t, v in tool_totals.items() if v / total_calls_all >= 0.01]
minor_tools = [t for t, v in tool_totals.items() if v / total_calls_all < 0.01]

if minor_tools:
    traj_df['tool_Other (<1%)'] = sum(traj_df[f'tool_{t}'] for t in minor_tools)
    minor_label = 'Other: ' + ', '.join(minor_tools)
    plot_tool_cols = [f'tool_{t}' for t in major_tools] + ['tool_Other (<1%)']
    plot_tool_names = major_tools + [minor_label]
else:
    plot_tool_cols = tool_cols
    plot_tool_names = tool_names

tool_palette = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6', '#1ABC9C', '#E67E22', '#34495E']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Panel 1: Mean tool calls per tool
ax = axes[0]
bar_width = 0.15
x = np.arange(len(MODEL_ORDER))
for j, (col, tname) in enumerate(zip(plot_tool_cols, plot_tool_names)):
    offset = (j - len(plot_tool_cols)/2 + 0.5) * bar_width
    means = [traj_df[traj_df['model'] == m][col].mean() for m in MODEL_ORDER]
    color = tool_palette[j % len(tool_palette)]
    ax.bar(x + offset, means, bar_width, color=color, edgecolor='black', linewidth=0.3, alpha=0.85)
    for i, v in enumerate(means):
        if v > 0.3:
            ax.text(x[i] + offset, v + 0.1, f'{v:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
tool_handles = [mpatches.Patch(facecolor=tool_palette[j % len(tool_palette)], edgecolor='black',
                                alpha=0.85, label=tname) for j, tname in enumerate(plot_tool_names)]
ax.legend(handles=tool_handles, fontsize=9, loc='upper right')
ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER, fontsize=9, rotation=15, ha='right')
ax.set_ylabel('Mean Tool Calls per Sample')
ax.set_title('Tool Usage: Absolute Count', fontweight='bold')

# Panel 2: Tool call ratios (stacked bar)
ax = axes[1]
bottoms = np.zeros(len(MODEL_ORDER))
for j, (col, tname) in enumerate(zip(plot_tool_cols, plot_tool_names)):
    ratios = []
    for m in MODEL_ORDER:
        mdf = traj_df[traj_df['model'] == m]
        total = mdf['n_tool_calls'].sum()
        ratios.append(mdf[col].sum() / total if total > 0 else 0)
    color = tool_palette[j % len(tool_palette)]
    ax.bar(x, ratios, color=color, edgecolor='white', linewidth=0.5, bottom=bottoms, alpha=0.85)
    for i, r in enumerate(ratios):
        if r > 0.05:
            ax.text(x[i], bottoms[i] + r/2, f'{r:.0%}', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')
    bottoms += ratios
ax.legend(handles=tool_handles, fontsize=9, loc='upper right')
ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER, fontsize=9, rotation=15, ha='right')
ax.set_ylabel('Fraction of Tool Calls')
ax.set_title('Tool Usage: Call Ratio', fontweight='bold')
ax.set_ylim(0, 1.05)

fig.suptitle(f'Tool Usage Analysis — {DOMAIN_NAME}', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'tool_usage_insane.png', bbox_inches='tight')
plt.show()

if minor_tools:
    print(f'\nTools aggregated into Other: {", ".join(minor_tools)}')

---

## 11. Time Efficiency & Reward Per Tool Call (Insane Mode)

With lower scores but potentially similar effort, efficiency metrics should drop in insane mode.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 11: Time efficiency and reward per tool call (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
n_panels = 3 if N_SAMPLES >= 3 else 2
fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 6))

# Panel 1: Reward per tool call
ax = axes[0]
reward_per_call = []
for m in MODEL_ORDER:
    mdf = traj_df[traj_df['model'] == m]
    mean_calls = mdf['n_tool_calls'].mean()
    rpc = mdf['score'].mean() / mean_calls if mean_calls > 0 else 0
    reward_per_call.append(rpc)
for i, (m, rpc) in enumerate(zip(MODEL_ORDER, reward_per_call)):
    ax.bar(i, rpc, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.text(i, rpc + max(reward_per_call)*0.02, f'{rpc:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels(MODEL_ORDER, fontsize=9, rotation=15, ha='right')
ax.set_ylabel('Mean Score / Mean Tool Calls')
ax.set_title('Reward per Tool Call', fontweight='bold')
ax.set_ylim(0, max(reward_per_call) * 1.25)

# Panel 2: Reward per minute
ax = axes[1]
reward_per_min = []
for m in MODEL_ORDER:
    mdf = traj_df[traj_df['model'] == m]
    mean_time = mdf['total_time'].mean() / 60
    rpm = mdf['score'].mean() / mean_time if mean_time > 0 else 0
    reward_per_min.append(rpm)
for i, (m, rpm) in enumerate(zip(MODEL_ORDER, reward_per_min)):
    ax.bar(i, rpm, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.text(i, rpm + max(reward_per_min)*0.02, f'{rpm:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels(MODEL_ORDER, fontsize=9, rotation=15, ha='right')
ax.set_ylabel('Mean Score / Mean Minutes')
ax.set_title('Reward per Minute', fontweight='bold')
ax.set_ylim(0, max(reward_per_min) * 1.25)

# Panel 3: Time per sample distributions
if N_SAMPLES >= 3:
    ax = axes[2]
    for i, m in enumerate(MODEL_ORDER):
        data = traj_df[traj_df['model'] == m]['total_time'].values / 60
        vp = ax.violinplot([data], positions=[i], showmeans=True, showmedians=True, widths=0.7)
        for pc in vp['bodies']:
            pc.set_facecolor(COLORS[m])
            pc.set_alpha(0.7)
        for part in ('cmaxes', 'cmins', 'cbars', 'cmeans', 'cmedians'):
            if part in vp:
                vp[part].set_edgecolor('black')
                vp[part].set_linewidth(0.8)
        ax.text(i, data.max() + 0.2, f'μ={np.mean(data):.1f}m', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels(MODEL_ORDER, fontsize=9, rotation=15, ha='right')
    ax.set_ylabel('Time (minutes)')
    ax.set_title('Time per Sample Distribution', fontweight='bold')

fig.suptitle(f'Agent Efficiency — {DOMAIN_NAME}', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'agent_efficiency_insane.png', bbox_inches='tight')
plt.show()

# Summary table
print('═══ Efficiency Summary (insane mode) ═══')
print(f'{"Model":25s} {"Score":>6s} {"Calls":>7s} {"Time":>7s} {"Rew/Call":>10s} {"Rew/Min":>10s}')
print('─' * 70)
for m, rpc, rpm in zip(MODEL_ORDER, reward_per_call, reward_per_min):
    mdf = traj_df[traj_df['model'] == m]
    print(f'{m:25s} {mdf["score"].mean():6.3f} {mdf["n_tool_calls"].mean():7.1f} '
          f'{mdf["total_time"].mean()/60:6.1f}m {rpc:10.4f} {rpm:10.4f}')

---

## 12. Score Outcome Segmentation by Effort (Insane Mode)

With missing tables, the zero-score fraction should increase across all effort quartiles.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 12: Score outcome segmentation by effort quartile (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
if N_SAMPLES < 8:
    print(f'⚠ Only {N_SAMPLES} sample(s) per model — skipping effort quartile analysis')
    for m in MODEL_ORDER:
        mdf = traj_df[traj_df['model'] == m]
        for _, row in mdf.iterrows():
            outcome = 'Perfect' if row['score'] >= 1.0 - 1e-6 else ('Zero' if row['score'] <= 1e-6 else 'Partial')
            print(f'  {m:25s}  calls={int(row["n_tool_calls"]):2d}  score={row["score"]:.3f}  → {outcome}')
else:
    outcome_colors = {'Perfect (1.0)': '#27AE60', 'Partial (0<s<1)': '#F39C12', 'Zero (0.0)': '#E74C3C'}
    n_models = len(MODEL_ORDER)
    fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 5), sharey=True)
    if n_models == 1:
        axes = [axes]

    for ax, m in zip(axes, MODEL_ORDER):
        mdf = traj_df[traj_df['model'] == m].copy()
        mdf['effort_q'] = pd.qcut(mdf['n_tool_calls'], q=4, labels=['Q1\n(low)', 'Q2', 'Q3', 'Q4\n(high)'], duplicates='drop')

        mdf['outcome'] = 'Partial (0<s<1)'
        mdf.loc[mdf['score'] >= 1.0 - 1e-6, 'outcome'] = 'Perfect (1.0)'
        mdf.loc[mdf['score'] <= 1e-6, 'outcome'] = 'Zero (0.0)'

        ct = pd.crosstab(mdf['effort_q'], mdf['outcome'], normalize='index')
        ct = ct.reindex(columns=['Perfect (1.0)', 'Partial (0<s<1)', 'Zero (0.0)'], fill_value=0)

        bottoms = np.zeros(len(ct))
        for outcome_cat in ['Perfect (1.0)', 'Partial (0<s<1)', 'Zero (0.0)']:
            if outcome_cat not in ct.columns:
                continue
            vals = ct[outcome_cat].values
            ax.bar(range(len(ct)), vals, bottom=bottoms, color=outcome_colors[outcome_cat],
                   edgecolor='white', linewidth=0.5, alpha=0.85)
            for xi, (v, b) in enumerate(zip(vals, bottoms)):
                if v > 0.05:
                    ax.text(xi, b + v/2, f'{v:.0%}', ha='center', va='center', fontsize=9, fontweight='bold')
            bottoms += vals

        call_ranges = mdf.groupby('effort_q')['n_tool_calls'].agg(['min', 'max'])
        xlabels = [f'{r.Index}\n({int(r.min)}–{int(r.max)} calls)' for r in call_ranges.itertuples()]
        ax.set_xticks(range(len(ct)))
        ax.set_xticklabels(xlabels, fontsize=8)
        ax.set_title(m, fontweight='bold', fontsize=11)
        ax.set_ylim(0, 1.05)

    axes[0].set_ylabel('Fraction of Samples')
    handles = [mpatches.Patch(facecolor=c, edgecolor='white', alpha=0.85, label=k) for k, c in outcome_colors.items()]
    fig.legend(handles=handles, loc='upper center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f'Score Outcomes by Tool-Call Effort Quartile — {DOMAIN_NAME}', fontweight='bold', fontsize=14, y=1.08)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / 'effort_outcome_segmentation_insane.png', bbox_inches='tight')
    plt.show()

---

## 13. Cross-Model Difficulty Agreement (Insane Mode)

Do models agree more or less on task difficulty when alert tables are stripped?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 13: Cross-model difficulty agreement (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr

if N_SAMPLES < 3:
    print(f'⚠ Only {N_SAMPLES} sample(s) — skipping correlation analysis')
    for m in MODEL_ORDER:
        mdf = traj_df[traj_df['model'] == m]
        print(f'  {m:25s}  score={mdf["score"].mean():.3f}')
else:
    score_pivot = samples_df.pivot(index='sample_id', columns='model', values='score').reindex(columns=MODEL_ORDER)

    n = len(MODEL_ORDER)
    corr_matrix = np.ones((n, n))
    p_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            m1, m2 = MODEL_ORDER[i], MODEL_ORDER[j]
            valid = score_pivot[[m1, m2]].dropna()
            if len(valid) > 2:
                rho, p = spearmanr(valid[m1], valid[m2])
                corr_matrix[i, j] = corr_matrix[j, i] = rho
                p_matrix[i, j] = p_matrix[j, i] = p

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='equal')

    for i in range(n):
        for j in range(n):
            val = corr_matrix[i, j]
            text_color = 'white' if val < 0.4 or val > 0.85 else 'black'
            sig = '' if i == j else ('*' if p_matrix[i, j] < 0.05 else '')
            ax.text(j, i, f'{val:.3f}{sig}', ha='center', va='center',
                    fontsize=11, fontweight='bold', color=text_color)

    ax.set_xticks(range(n)); ax.set_xticklabels(MODEL_ORDER, rotation=30, ha='right', fontsize=10)
    ax.set_yticks(range(n)); ax.set_yticklabels(MODEL_ORDER, fontsize=10)
    ax.set_title(f'Cross-Model Score Correlation (Spearman ρ) — {DOMAIN_NAME}', fontweight='bold', pad=15)
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Spearman ρ', fontsize=10)
    fig.text(0.5, -0.02, '* = p < 0.05. Higher ρ = models agree more on which tasks are hard.',
             ha='center', fontsize=9, fontstyle='italic', color='gray')
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / 'cross_model_correlation_insane.png', bbox_inches='tight')
    plt.show()

---

## 14. SQL Query Analysis (Insane Mode)

### Key questions for insane mode

- Do agents produce **more errors** when querying stripped tables (SecurityAlert, AlertEvidence, etc.)?
- Does the **empty result rate** spike as agents try to query tables that no longer exist?
- Do models adapt by querying alternative tables, or do they keep retrying the missing ones?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 14 (EXCYTIN-SPECIFIC): SQL Query Analysis (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
import re
import zipfile
import json as _json
from inspect_ai.log import read_eval_log_sample

# SQL detection patterns
SQL_PATTERN = re.compile(
    r'\b(SELECT|INSERT|UPDATE|DELETE|CREATE|DROP|ALTER|SHOW|DESCRIBE|EXPLAIN|USE|WITH)\b',
    re.IGNORECASE
)
ERROR_PATTERNS = re.compile(
    r"(ERROR\s+\d+|error:|syntax error|no such table|unknown column|access denied|doesn.t exist|command not found|permission denied|You have an error)",
    re.IGNORECASE
)
EMPTY_PATTERNS = re.compile(r'(Empty set|0 rows|\(0 rows\)|^\s*$)')
ROW_COUNT_PATTERN = re.compile(r'(\d+) rows? in set')

def classify_query_result(result_text: str) -> str:
    # Classify a SQL query result into success/error/empty/large
    if not result_text or result_text.strip() == '':
        return 'empty'
    if ERROR_PATTERNS.search(result_text):
        return 'error'
    if EMPTY_PATTERNS.search(result_text.strip()):
        return 'empty'
    m = ROW_COUNT_PATTERN.search(result_text)
    if m:
        n_rows = int(m.group(1))
        if n_rows == 0:
            return 'empty'
        elif n_rows > 100:
            return 'large_result'
        else:
            return 'success'
    content_lines = [l for l in result_text.strip().split('\n') if l.strip() and not l.startswith('+')]
    if len(content_lines) <= 2:
        return 'empty'
    elif len(content_lines) > 102:
        return 'large_result'
    else:
        return 'success'

def extract_sql_queries(path: str, sample_id: str) -> list[dict]:
    # Extract SQL queries and results from a sample
    queries = []
    try:
        sample = read_eval_log_sample(path, id=sample_id)
        messages = sample.messages
        for i, msg in enumerate(messages):
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    if tc.function != 'bash':
                        continue
                    cmd = ''
                    if isinstance(tc.arguments, dict):
                        cmd = tc.arguments.get('cmd', '') or tc.arguments.get('command', '') or str(tc.arguments)
                    elif isinstance(tc.arguments, str):
                        cmd = tc.arguments
                    if not SQL_PATTERN.search(cmd):
                        continue
                    result_text = ''
                    for j in range(i+1, min(i+5, len(messages))):
                        rmsg = messages[j]
                        if hasattr(rmsg, 'tool_call_id') and rmsg.tool_call_id == tc.id:
                            if isinstance(rmsg.content, str):
                                result_text = rmsg.content
                            elif isinstance(rmsg.content, list):
                                result_text = ' '.join(
                                    getattr(c, 'text', str(c)) for c in rmsg.content
                                )
                            break
                    queries.append({
                        'command': cmd[:500],
                        'result_class': classify_query_result(result_text),
                        'result_length': len(result_text),
                    })
    except Exception:
        try:
            with zipfile.ZipFile(path) as z:
                for name in z.namelist():
                    if not (name.startswith('samples/') and name.endswith('.json')):
                        continue
                    raw = _json.loads(z.read(name))
                    if str(raw.get('id', '')) != sample_id:
                        continue
                    messages_raw = raw.get('messages', [])
                    for i, msg in enumerate(messages_raw):
                        if msg.get('role') != 'assistant':
                            continue
                        for tc in msg.get('tool_calls', []):
                            if tc.get('function', '') != 'bash':
                                continue
                            args = tc.get('arguments', {})
                            cmd = args.get('cmd', '') or args.get('command', '') or str(args)
                            if not SQL_PATTERN.search(cmd):
                                continue
                            result_text = ''
                            tc_id = tc.get('id', '')
                            for j in range(i+1, min(i+5, len(messages_raw))):
                                rmsg = messages_raw[j]
                                if rmsg.get('tool_call_id') == tc_id:
                                    content = rmsg.get('content', '')
                                    result_text = content if isinstance(content, str) else str(content)
                                    break
                            queries.append({
                                'command': cmd[:500],
                                'result_class': classify_query_result(result_text),
                                'result_length': len(result_text),
                            })
                    break
        except Exception:
            pass
    return queries

# Extract SQL queries from all samples
print('Extracting SQL queries from all samples (this may take a minute)...')
sql_rows = []
for model_name, log_file in EVAL_LOGS.items():
    path = os.path.join(LOG_DIR, log_file)
    sample_ids = traj_df[traj_df['model'] == model_name]['sample_id'].tolist()
    for sid in sample_ids:
        for q in extract_sql_queries(path, sid):
            q['model'] = model_name
            q['sample_id'] = sid
            sql_rows.append(q)

sql_df = pd.DataFrame(sql_rows)
print(f'✓ Extracted {len(sql_df)} SQL queries across {sql_df["model"].nunique()} models')
print()

# Summary per model
print('═══ SQL Query Summary per Model (insane mode) ═══')
print(f'{"Model":25s} {"Total":>6s} {"Success":>8s} {"Empty":>8s} {"Error":>8s} {"Large":>8s} {"Queries/Sample":>15s}')
print('─' * 85)
for m in MODEL_ORDER:
    mq = sql_df[sql_df['model'] == m]
    n_samples = len(traj_df[traj_df['model'] == m])
    total = len(mq)
    success = (mq['result_class'] == 'success').sum()
    empty = (mq['result_class'] == 'empty').sum()
    error = (mq['result_class'] == 'error').sum()
    large = (mq['result_class'] == 'large_result').sum()
    per_sample = total / n_samples if n_samples > 0 else 0
    print(f'{m:25s} {total:6d} {success:7d} ({success/total*100 if total else 0:4.1f}%) '
          f'{empty:5d} ({empty/total*100 if total else 0:4.1f}%) '
          f'{error:5d} ({error/total*100 if total else 0:4.1f}%) '
          f'{large:5d} ({large/total*100 if total else 0:4.1f}%) '
          f'{per_sample:14.1f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 14b: SQL Query Outcome Visualization (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
result_classes = ['success', 'empty', 'error', 'large_result']
result_labels = ['Success (≤100 rows)', 'Empty Result', 'Error', 'Large Result (>100 rows)']
result_colors = ['#27AE60', '#F39C12', '#E74C3C', '#3498DB']

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# Panel 1: Query outcome rate per model (stacked bar)
ax = axes[0]
x = np.arange(len(MODEL_ORDER))
bottoms = np.zeros(len(MODEL_ORDER))
handles = []
for rc, label, color in zip(result_classes, result_labels, result_colors):
    rates = []
    for m in MODEL_ORDER:
        mq = sql_df[sql_df['model'] == m]
        rates.append((mq['result_class'] == rc).sum() / len(mq) if len(mq) > 0 else 0)
    bars = ax.bar(x, rates, bottom=bottoms, color=color, edgecolor='white', linewidth=0.5, alpha=0.85)
    handles.append(mpatches.Patch(facecolor=color, edgecolor='white', alpha=0.85, label=label))
    for i, (r, b) in enumerate(zip(rates, bottoms)):
        if r > 0.04:
            ax.text(x[i], b + r/2, f'{r:.0%}', ha='center', va='center', fontsize=9, fontweight='bold')
    bottoms += rates
ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER, fontsize=9, rotation=15, ha='right')
ax.set_ylabel('Fraction of SQL Queries')
ax.set_title('Query Outcome Rate', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(handles=handles, fontsize=8, loc='upper right')

# Panel 2: Queries per sample (bar)
ax = axes[1]
queries_per_sample = []
for m in MODEL_ORDER:
    n_q = len(sql_df[sql_df['model'] == m])
    n_s = len(traj_df[traj_df['model'] == m])
    queries_per_sample.append(n_q / n_s if n_s > 0 else 0)
for i, (m, qps) in enumerate(zip(MODEL_ORDER, queries_per_sample)):
    ax.bar(i, qps, color=COLORS[m], edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.text(i, qps + max(queries_per_sample)*0.02, f'{qps:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels(MODEL_ORDER, fontsize=9, rotation=15, ha='right')
ax.set_ylabel('SQL Queries per Sample')
ax.set_title('Query Volume', fontweight='bold')
ax.set_ylim(0, max(queries_per_sample) * 1.25)

# Panel 3: Success rate vs. overall score (scatter)
ax = axes[2]
for i, m in enumerate(MODEL_ORDER):
    mq = sql_df[sql_df['model'] == m]
    success_rate = (mq['result_class'] == 'success').sum() / len(mq) if len(mq) > 0 else 0
    mean_score = traj_df[traj_df['model'] == m]['score'].mean()
    ax.scatter(success_rate, mean_score, color=COLORS[m], s=200, edgecolor='black', linewidth=1.5, zorder=5)
    ax.annotate(m.split()[0] + ' ' + m.split()[-1], (success_rate, mean_score),
                textcoords='offset points', xytext=(8, 5), fontsize=9, fontweight='bold')
ax.set_xlabel('SQL Query Success Rate')
ax.set_ylabel('Mean Overall Score')
ax.set_title('Success Rate vs. Score', fontweight='bold')
ax.grid(alpha=0.3)

fig.suptitle(f'SQL Query Analysis — {DOMAIN_NAME}', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'sql_query_analysis_insane.png', bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 14c: SQL Query Outcome vs. Score per Sample (insane mode)
# ══════════════════════════════════════════════════════════════════════════════
sample_query_stats = sql_df.groupby(['model', 'sample_id']).apply(
    lambda g: pd.Series({
        'n_queries': len(g),
        'success_rate': (g['result_class'] == 'success').mean(),
        'error_rate': (g['result_class'] == 'error').mean(),
        'empty_rate': (g['result_class'] == 'empty').mean(),
    })
).reset_index()

sample_query_stats = sample_query_stats.merge(
    traj_df[['model', 'sample_id', 'score', 'n_tool_calls']],
    on=['model', 'sample_id']
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Error rate vs. score (binned)
ax = axes[0]
for m in MODEL_ORDER:
    mdf = sample_query_stats[sample_query_stats['model'] == m]
    if len(mdf) < 5:
        continue
    bins = pd.cut(mdf['error_rate'], bins=5)
    grouped = mdf.groupby(bins, observed=True)['score'].mean()
    midpoints = [interval.mid for interval in grouped.index]
    ax.plot(midpoints, grouped.values, color=COLORS[m], marker='o', linewidth=2, label=m, markersize=6)
ax.set_xlabel('SQL Error Rate (per sample)')
ax.set_ylabel('Mean Score')
ax.set_title('Error Rate Impact on Score', fontweight='bold')
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.3)

# Panel 2: Query volume vs. score (binned)
ax = axes[1]
for m in MODEL_ORDER:
    mdf = sample_query_stats[sample_query_stats['model'] == m]
    if len(mdf) < 5:
        continue
    bins = pd.cut(mdf['n_queries'], bins=5)
    grouped = mdf.groupby(bins, observed=True)['score'].mean()
    midpoints = [interval.mid for interval in grouped.index]
    ax.plot(midpoints, grouped.values, color=COLORS[m], marker='s', linewidth=2, label=m, markersize=6)
ax.set_xlabel('Number of SQL Queries (per sample)')
ax.set_ylabel('Mean Score')
ax.set_title('Query Volume vs. Score', fontweight='bold')
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.3)

fig.suptitle(f'SQL Query–Score Relationship — {DOMAIN_NAME}', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'sql_query_score_relationship_insane.png', bbox_inches='tight')
plt.show()

# Correlation summary
from scipy.stats import spearmanr as _spearmanr
print('═══ SQL Error Rate → Score Correlation (insane mode, per model) ═══')
for m in MODEL_ORDER:
    mdf = sample_query_stats[sample_query_stats['model'] == m]
    if len(mdf) > 2:
        rho, p = _spearmanr(mdf['error_rate'], mdf['score'])
        print(f'  {m:25s}  ρ={rho:+.3f}  p={p:.4f}  (n={len(mdf)} samples with SQL queries)')

---

## Key Takeaways (Insane Mode)

> **Fill in after running the notebook** — compare results with [excytin_analysis.ipynb](excytin_analysis.ipynb) (normal mode).

### Performance Impact
- How much did overall scores drop compared to normal mode?
- Which models were most resilient?
- Which incidents were most affected?

### Behavioral Changes
- Did models use more tool calls (searching for missing data)?
- Did SQL error rates spike (querying nonexistent tables)?
- Did empty result rates increase?

### Implications
- How dependent are current models on "easy path" alert tables?
- Does insane mode better differentiate model capabilities?
- Which models adapt best when primary data sources are unavailable?

---

## Next Steps

### Adding your own models (insane mode)

1. Run: `uv run inspect eval domains/excytin --model <model_id> -T mode=insane -T dataset=latest_test_set`
2. Copy the `.eval` file to `eval_samples/excytin/` with `_insane` suffix
3. Edit the **Configuration cell** above
4. Re-run the notebook

### Comparing with normal mode

Open [excytin_analysis.ipynb](excytin_analysis.ipynb) side-by-side to compare normal vs. insane mode results.